# 05 - Fine-tune Fine-Tashkeel (ByT5) for KSAA Dataset

**Base Model:** `basharalrfooh/Fine-Tashkeel` (ByT5 720M)
**Architecture:** Byte-level T5 Seq2Seq
**Reported Performance:** DER 0.95%, WER 2.49%

## Why ByT5 for Arabic Diacritization?
- **Byte-level processing**: No vocabulary limitations, handles rare words and dialects
- **Preserves morphology**: Arabic morphological patterns intact
- **Pre-trained on Tashkeela**: Already knows Arabic diacritization patterns

## Training Techniques:
1. **Curriculum Learning**: Short → Long sentences
2. **Label Smoothing**: 0.1 for better generalization
3. **Data Augmentation**: Word dropout, character noise
4. **Post-processing**: Diacritic correction rules
5. **Mixed Precision (FP16)**: Faster training

## Tasks:
1. Load and preprocess KSAA training data
2. Fine-tune model with curriculum learning
3. Evaluate on dev set
4. Generate test submission

In [1]:
!pip install -q transformers torch accelerate datasets tqdm pandas

In [2]:
# Setup
import os, sys, json, re, torch, gc, zipfile
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Environment detection
IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))

# Import utilities (includes safeguards for empty responses & separated characters)
from utils_diacritization import (
    normalize_arabic, remove_diacritics, postprocess_diacritics,
    compute_metrics, print_metrics, sort_by_length, augment_sample,
    is_valid_output, repair_output, get_safe_seq2seq_generation_config
)

# Paths
DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
TRAIN_DIR = DATA_DIR / 'Train'
DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
TEST_DIR = PROJECT_DIR / 'Test'
TEST_INPUT = TEST_DIR / 'test_input.json'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
SUBMISSION_DIR = PROJECT_DIR / 'submissions'
CHECKPOINTS_DIR = PROJECT_DIR / 'checkpoints' / 'fine_tashkeel_ft'

for d in [OUTPUT_DIR, SUBMISSION_DIR, CHECKPOINTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment: 'Local' | Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")
    torch.backends.cuda.matmul.allow_tf32 = True

def clear_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
        print(f"GPU Memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB allocated")

Environment: Local | Device: cuda
GPU: NVIDIA RTX A5000 (23.6 GB)


## 1. Load and Preprocess Training Data

In [3]:
# Load training data from TSV + text files
train_tsv = TRAIN_DIR / 'train_list.tsv'
train_df = pd.read_csv(train_tsv, sep='\t')
print(f"Training samples in TSV: {len(train_df)}")

def load_training_data():
    """Load training data with preprocessing."""
    data = []
    text_dir = TRAIN_DIR / 'train_text'
    
    for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Loading data"):
        txt_file = text_dir / f"{row['stem']}.txt"
        if txt_file.exists():
            with open(txt_file, 'r', encoding='utf-8') as f:
                diacritized = f.read().strip()
            
            # Preprocess
            diacritized = normalize_arabic(diacritized)
            undiacritized = remove_diacritics(diacritized)
            
            # Skip empty or too short samples
            if len(undiacritized.split()) < 2:
                continue
                
            data.append({
                'id': row['stem'],
                'text_undiacritized': undiacritized,
                'text_diacritized': diacritized
            })
    return data

train_data = load_training_data()
print(f"Loaded {len(train_data)} training samples")

# Sort by length for curriculum learning
train_data = sort_by_length(train_data)
print(f"Sorted by length: shortest={len(train_data[0]['text_undiacritized'])} chars, "
      f"longest={len(train_data[-1]['text_undiacritized'])} chars")

Training samples in TSV: 2327


Loading data: 100%|██████████| 2327/2327 [00:06<00:00, 376.35it/s]

Loaded 2317 training samples
Sorted by length: shortest=4 chars, longest=241 chars


In [4]:
# Data augmentation - create augmented copies
import random
random.seed(42)

augmented_data = []
for sample in train_data:
    # Original sample
    augmented_data.append(sample)
    
    # Create 1-2 augmented versions
    for _ in range(random.randint(1, 2)):
        aug_in, aug_out = augment_sample(
            sample['text_undiacritized'],
            sample['text_diacritized'],
            augment_prob=0.5
        )
        if aug_in != sample['text_undiacritized']:  # Only if actually augmented
            augmented_data.append({
                'id': f"{sample['id']}_aug",
                'text_undiacritized': aug_in,
                'text_diacritized': aug_out
            })

print(f"After augmentation: {len(augmented_data)} samples (original: {len(train_data)})")

# Shuffle augmented data while keeping curriculum order for original
random.shuffle(augmented_data)
train_data = augmented_data

After augmentation: 3523 samples (original: 2317)


## 2. Load Model and Tokenizer

In [5]:
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)
from datasets import Dataset

MODEL_NAME = 'basharalrfooh/Fine-Tashkeel'
MODEL_KEY = 'fine_tashkeel_ft'

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,  # Full precision for training stability
).to(device)

# Enable gradient checkpointing to save memory
model.gradient_checkpointing_enable()

print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} parameters")
clear_gpu()

Loading basharalrfooh/Fine-Tashkeel...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/308 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded: 720,870,144 parameters
Trainable: 720,870,144 parameters
GPU Memory: 2.75 GB allocated


## 3. Prepare Dataset

In [6]:
MAX_LENGTH = 512

def preprocess_function(examples):
    """Tokenize inputs and targets."""
    model_inputs = tokenizer(
        examples['text_undiacritized'],
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length'
    )
    
    labels = tokenizer(
        examples['text_diacritized'],
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length'
    )
    
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Create training dataset
train_dataset = Dataset.from_list(train_data)
train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=['id', 'text_undiacritized', 'text_diacritized'],
    desc="Tokenizing training data"
)

# Load and prepare dev data
with open(DEV_INPUT, 'r', encoding='utf-8') as f:
    dev_input = json.load(f)
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f:
    dev_output = json.load(f)

dev_lookup = {item['id']: item['text_diacritized'] for item in dev_output}
dev_data = [
    {
        'text_undiacritized': normalize_arabic(item['text_undiacritized']),
        'text_diacritized': normalize_arabic(dev_lookup.get(item['id'], ''))
    }
    for item in dev_input if item['id'] in dev_lookup
]

eval_dataset = Dataset.from_list(dev_data)
eval_dataset = eval_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=['text_undiacritized', 'text_diacritized'],
    desc="Tokenizing eval data"
)

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

Tokenizing training data:   0%|          | 0/3523 [00:00<?, ? examples/s]

Tokenizing eval data:   0%|          | 0/260 [00:00<?, ? examples/s]

Train: 3523, Eval: 260


## 4. Training Configuration

In [7]:
# Training arguments - Optimized for 24GB GPU (720M model needs more memory)
training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINTS_DIR),
    
    # Training params - Optimized for 24GB GPU
    num_train_epochs=5,
    per_device_train_batch_size=6,   # Balanced for 720M model
    per_device_eval_batch_size=8,    # Can use more for eval
    gradient_accumulation_steps=5,   # Effective batch size: 30
    
    # Optimizer
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    
    # Regularization
    label_smoothing_factor=0.1,
    
    # Mixed precision - FP16 for speed
    fp16=True,
    
    # Evaluation & Saving
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Logging
    logging_steps=50,
    report_to="none",
    
    # Generation
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    
    # Performance
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accum: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16: {training_args.fp16}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training configuration:
  Epochs: 5
  Batch size: 6
  Gradient accum: 5
  Effective batch: 30
  Learning rate: 2e-05
  FP16: True


## 5. Train the Model

In [ ]:
# Initialize trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# Check for existing checkpoints
clear_gpu()
checkpoint = None
checkpoints = list(CHECKPOINTS_DIR.glob('checkpoint-*'))
if checkpoints:
    checkpoint = str(max(checkpoints, key=lambda x: int(x.name.split('-')[1])))
    print(f"Resuming from checkpoint: {checkpoint}")

# Train!
print("Starting training...")
trainer.train(resume_from_checkpoint=checkpoint)

GPU Memory: 2.75 GB allocated
Starting training...


Step,Training Loss,Validation Loss
100,1825.513125,42.127571


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Save the final model
final_path = CHECKPOINTS_DIR / 'final'
trainer.save_model(str(final_path))
tokenizer.save_pretrained(str(final_path))
print(f"Model saved to: {final_path}")
clear_gpu()

In [ ]:
 clear_gpu()

## 6. Inference Functions

In [ ]:
# Load the fine-tuned model for inference (FP16 for speed)
final_model_path = CHECKPOINTS_DIR / 'final'
if final_model_path.exists():
    print(f"Loading fine-tuned model from {final_model_path}")
    # Clear training model first
    del model
    clear_gpu()
    
    model = AutoModelForSeq2SeqLM.from_pretrained(
        final_model_path,
        torch_dtype=torch.float16,  # FP16 for faster inference
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(final_model_path)
else:
    print("Using model from training (already in memory)")
    model = model.half()  # Convert to FP16

model.eval()
print("Model ready for inference (FP16)")

In [ ]:
@torch.inference_mode()
def diacritize(text: str, apply_postprocess: bool = True) -> str:
    """
    Diacritize Arabic text using the fine-tuned model.
    Includes safeguards against empty responses and separated characters.
    
    Args:
        text: Undiacritized Arabic text
        apply_postprocess: Whether to apply post-processing rules
    
    Returns:
        Diacritized text (or original text if model fails)
    """
    original_text = text
    
    # Preprocess
    text = normalize_arabic(text)
    
    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=MAX_LENGTH,
        truncation=True
    ).to(device)
    
    # Get safe generation config
    gen_config = get_safe_seq2seq_generation_config()
    gen_config['max_length'] = MAX_LENGTH
    
    # Generate with safe parameters
    outputs = model.generate(
        **inputs,
        **gen_config
    )
    
    # Decode
    result = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    
    # Repair output (fixes separated characters, validates, falls back to input if invalid)
    if apply_postprocess:
        result = repair_output(result, original_text, fallback_to_input=True)
    
    return result

## 7. Evaluate on Dev Set

In [ ]:
# Checkpoint support
CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_checkpoint.json"

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_checkpoint(predictions):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({
            'processed_ids': [p['id'] for p in predictions],
            'predictions': predictions
        }, f, ensure_ascii=False)

# Run inference on dev set
processed_ids, dev_predictions = load_checkpoint()
print(f"Dev: {len(dev_input)} samples, {len(processed_ids)} already done")

for item in tqdm(dev_input, desc="Dev Set"):
    if item['id'] in processed_ids:
        continue
    try:
        result = diacritize(item['text_undiacritized'])
        dev_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_checkpoint(dev_predictions)
    except Exception as e:
        print(f"Error on {item['id']}: {e}")
        dev_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_checkpoint(dev_predictions)

# Save predictions
with open(OUTPUT_DIR / f"{MODEL_KEY}_dev_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=2)

In [ ]:
# Compute metrics
print("\n" + "="*60)
print("DEV SET RESULTS - With Post-processing")
print("="*60)

metrics_with_irab = compute_metrics(dev_predictions, dev_output, exclude_irab=False)
print(f"\n[Including I'rab (case endings)]")
print(f"  DER: {metrics_with_irab['DER']*100:.2f}%")
print(f"  WER: {metrics_with_irab['WER']*100:.2f}% (PRIMARY)")
print(f"  SER: {metrics_with_irab['SER']*100:.2f}%")

metrics_no_irab = compute_metrics(dev_predictions, dev_output, exclude_irab=True)
print(f"\n[Excluding I'rab (case endings)]")
print(f"  DER: {metrics_no_irab['DER']*100:.2f}%")
print(f"  WER: {metrics_no_irab['WER']*100:.2f}%")
print(f"  SER: {metrics_no_irab['SER']*100:.2f}%")

## 8. Generate Test Submission

In [ ]:
# Load test data
with open(TEST_INPUT, 'r', encoding='utf-8') as f:
    test_input = json.load(f)

TEST_CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_test_checkpoint.json"

def load_test_checkpoint():
    if TEST_CHECKPOINT_FILE.exists():
        with open(TEST_CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_test_checkpoint(predictions):
    with open(TEST_CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({
            'processed_ids': [p['id'] for p in predictions],
            'predictions': predictions
        }, f, ensure_ascii=False)

# Run inference on test set
test_processed_ids, test_predictions = load_test_checkpoint()
print(f"Test: {len(test_input)} samples, {len(test_processed_ids)} already done")

for item in tqdm(test_input, desc="Test Set"):
    if item['id'] in test_processed_ids:
        continue
    try:
        result = diacritize(item['text_undiacritized'])
        test_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_test_checkpoint(test_predictions)
    except Exception as e:
        test_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_test_checkpoint(test_predictions)

# Save predictions
with open(OUTPUT_DIR / f"{MODEL_KEY}_test_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

In [ ]:
# Create submission ZIP
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

json_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

zip_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission_{timestamp}.zip"
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')

print("="*60)
print("SUBMISSION READY FOR CODABENCH")
print("="*60)
print(f"ZIP:  {zip_file}")
print(f"Size: {zip_file.stat().st_size / 1024:.1f} KB")
print(f"Samples: {len(test_predictions)}")

In [ ]:
# Google Drive sync removed for public release
import subprocess
sync_script = PROJECT_DIR / 'sync_results.sh'
if sync_script.exists() and False:
    print("Syncing to Google Drive...")
    subprocess.run(['bash', str(sync_script)])

# Final cleanup - free GPU memory
print("\nCleaning up GPU memory...")
del model
del tokenizer
clear_gpu()
print("Done! GPU memory released.")

## 9. Sample Comparisons

In [ ]:
# Show sample comparisons
gt_lookup = {item['id']: item['text_diacritized'] for item in dev_output}

print("Sample Comparisons (Dev Set):")
print("="*80)

for i, pred in enumerate(dev_predictions[:5]):
    print(f"\n[{i+1}] ID: {pred['id']}")
    print(f"Input:  {remove_diacritics(pred['text_diacritized'])[:60]}...")
    print(f"Pred:   {pred['text_diacritized'][:60]}...")
    print(f"Ground: {gt_lookup.get(pred['id'], 'N/A')[:60]}...")